In [ ]:
#from google.colab import files
#uploaded = files.upload()
#fname = next(iter(uploaded))
#fname


In [ ]:
import re
import pandas as pd
import numpy as np

def parse_wrapped_file(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = [ln.rstrip("\n") for ln in f]

    # -------- 1) collect header indices (idx … plus wrapped header lines) --------
    try:
        idx_start = next(i for i, ln in enumerate(lines) if ln.strip().startswith("idx"))
    except StopIteration:
        raise ValueError("Couldn't find a line starting with 'idx'.")

    idx_cols = []
    k = idx_start
    while k < len(lines):
        ln = lines[k]
        if "|" in ln:                      # header ends when first metadata line appears
            break
        idx_cols.extend(re.findall(r"\d+", ln))
        k += 1

    if not idx_cols:
        raise ValueError("No numeric column indices found after 'idx'.")
    n_vals = len(idx_cols)

    # -------- 2) parse records (tolerant to missing / extra values) --------
    num_re = re.compile(r"[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?")
    records = []
    padded_ct = 0
    truncated_ct = 0

    # find all metadata line positions up front
    meta_pos = [i for i in range(k, len(lines)) if "|" in lines[i]]
    meta_pos.append(len(lines))  # sentinel for last block

    for m in range(len(meta_pos) - 1):
        i = meta_pos[m]
        nxt = meta_pos[m + 1]  # first line of the next record (or EOF)

        meta_line = lines[i]
        parts = [p.strip() for p in meta_line.split("|")]

        # Extract any numbers on the SAME line after the last '|'
        after_last = parts[-1] if parts else ""
        same_line_nums = num_re.findall(after_last)

        # Determine charge if the first token looks like an integer
        charge_token = None
        same_line_vals = same_line_nums
        if same_line_nums and re.fullmatch(r"[+-]?\d+", same_line_nums[0]):
            charge_token = same_line_nums[0]
            same_line_vals = same_line_nums[1:]

        # Clean last metadata field (remove nums we just captured)
        if parts:
            parts[-1] = re.sub(num_re, "", after_last).strip()

        # pad metadata to 5 fields
        parts += [""] * (5 - len(parts))
        gene, protein, site, peptide, charge = parts[:5]
        if charge_token is not None and (not charge or not re.search(r"\d", charge)):
            charge = charge_token

        # Collect values from current line (same-line) + following lines up to (but not including) nxt
        vals = [*same_line_vals]
        for row in range(i + 1, nxt):
            ln = lines[row]
            if not ln.strip():
                continue
            vals.extend(num_re.findall(ln))

        # Normalize length: pad with NaN if short; trim if long
        if len(vals) < n_vals:
            vals.extend([np.nan] * (n_vals - len(vals)))
            padded_ct += 1
        elif len(vals) > n_vals:
            vals = vals[:n_vals]
            truncated_ct += 1

        vals = [float(v) if v is not None and v == v else np.nan for v in vals]  # keep NaNs

        rec = {
            "gene": gene,
            "protein": protein,
            "site": site,
            "peptide": peptide,
            "charge": pd.to_numeric(charge, errors="coerce"),
        }
        rec.update(dict(zip(idx_cols, vals)))
        records.append(rec)

    df = pd.DataFrame.from_records(
        records, columns=["gene","protein","site","peptide","charge"] + idx_cols
    )

    if padded_ct or truncated_ct:
        print(f"Note: padded {padded_ct} record(s) with NaN and truncated {truncated_ct} record(s) to {n_vals} values.")
    return df

# Run it on your uploaded file from files.upload()
#df = parse_wrapped_file(fname)
#df.head(), df.shape


In [ ]:
#################
####################
############################ proteomic data
import re
import pandas as pd
import numpy as np

def parse_proteomics_wrapped(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = [ln.rstrip("\n") for ln in f]

    # 1) Find idx header start
    protein_start = next(i for i, ln in enumerate(lines) if ln.strip().startswith("protein"))

    # 2) Collect all sample IDs from idx header block (may wrap across lines)
    sample_ids = []
    k = protein_start
    while k < len(lines):
        ln = lines[k]
        # header ends at first line that looks like an ID row (e.g. ENSG...)
        if re.match(r"^\s*ENSG\d+", ln):
            break
        sample_ids += re.findall(r"\d+", ln)
        k += 1

    n_expected = len(sample_ids)
    if n_expected == 0:
        raise ValueError("No sample IDs found in idx header.")

    # regex for numbers (floats + scientific)
    num_re = re.compile(r"[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?")

    # 3) Identify row-start lines (protein IDs). Your file shows ENSG... at row start.
    row_starts = [i for i in range(k, len(lines)) if re.match(r"^\s*ENSG\d+", lines[i])]
    row_starts.append(len(lines))  # sentinel

    records = []
    padded = 0
    overflow = 0

    for r in range(len(row_starts) - 1):
        i = row_starts[r]
        j = row_starts[r + 1]

        # first token on the row-start line is the ID
        prot_id = lines[i].strip().split()[0]  # e.g., ENSG00000000003.15

        # collect all values until the next protein ID line
        vals = []
        # sometimes numbers can be on same line after the ID (rare, but safe)
        vals += num_re.findall(lines[i].replace(prot_id, " ", 1))

        for t in range(i + 1, j):
            vals += num_re.findall(lines[t])

        # normalize length to expected number of samples
        if len(vals) < n_expected:
            vals += [np.nan] * (n_expected - len(vals))
            padded += 1
        elif len(vals) > n_expected:
            # keep extras instead of truncating? for proteomics it's usually safe to truncate
            # but we can keep them in extra columns if you prefer
            vals = vals[:n_expected]
            overflow += 1

        records.append([prot_id] + [float(x) if x == x else np.nan for x in vals])

    cols = ["protein_id"] + sample_ids
    df = pd.DataFrame(records, columns=cols)

    print(f"Parsed {len(df)} proteins with {n_expected} samples.")
    if padded:
        print(f"Note: padded {padded} rows with NaN (missing values).")
    if overflow:
        print(f"Note: truncated {overflow} rows that had >{n_expected} values.")

    return df

# Example:
# df_prot = parse_proteomics_wrapped("HCC_proteome_wrapped.txt")
# df_prot.head()
# df_prot.to_parquet("HCC_proteome_clean.parquet", index=False)
# df_prot.to_csv("HCC_proteome_clean.csv", index=False)


In [ ]:
df = parse_proteomics_wrapped("HCC_RNAseq_gene_RSEM_coding_UQ_log2_Normal.txt")


Parsed 17734 proteins with 159 samples.


In [ ]:
df.head(), df.shape

(        protein_id       1014       1016       1022       1026       1028  \
 0  ENSG00000000003  11.436244  11.776734  12.254245  11.885214  12.210289   
 1  ENSG00000000005   0.082311   0.082311   0.082311   0.082311   3.505087   
 2  ENSG00000000419   9.687794   9.896091   9.963434   9.966906  10.291427   
 3  ENSG00000000457   9.138464   9.211131   8.974145   9.512108   8.950059   
 4  ENSG00000000460   6.015370   7.722897   7.592676   7.012119   7.790714   
 
         1032       1042       1044       1046  ...        954        956  \
 0  11.631354  12.308004  12.169839  12.151077  ...  11.841257  11.930509   
 1   0.082311   0.082311   3.187365   6.505572  ...   0.082311   4.347097   
 2   9.430738  10.245966   9.929580   9.847514  ...  10.014394   9.841271   
 3  10.239497   9.909570   9.825107   9.836664  ...   9.400615   9.379222   
 4   7.222266   6.956779   7.194377   7.660526  ...   6.737425   7.413976   
 
          958        964        966        968        976        9

In [ ]:
df.to_csv("clean_protein_normal.csv", index=False)
df.to_parquet("clean_protein_normal.parquet", index=False)
from google.colab import files
files.download("clean_protein_normal.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
df = pd.read_csv("clean.csv")          # or: pd.read_csv("your.tsv", sep="\t")
df.head()


,gene,protein,site,peptide,charge,112,113,123,125,127,...,1021,1025,1027,1031,1041,1043,1045,447,535,697
0,ENSG00000003056.8,ENSP00000000412.3,S267,DDQLGEESEERDDHL,1,23.337745,23.193014,22.593026,23.308922,23.373497,...,22.760057,23.782227,22.780906,23.091338,23.641987,22.772814,23.013947,23.132176,23.298945,23.502287
1,ENSG00000048028.11,ENSP00000003302.4,S1053,PPTIRPNSPYDLCSR,1,24.471825,25.622572,24.726388,25.520393,24.587026,...,25.421100,25.930739,25.179525,25.968220,25.869581,26.035344,24.699102,24.893689,25.508558,24.752528
2,ENSG00000004776.13,ENSP00000004982.3,S16,PSWLRRASAPLPGLS,1,28.075454,28.159920,28.429887,28.401917,28.614121,...,27.969410,27.782625,28.039739,27.638366,27.798161,27.855082,29.652364,28.842153,28.467127,28.992966
3,ENSG00000006453.14,ENSP00000005260.8,S261,VSGTPQASPMIERSN,1,25.863522,24.383697,23.472160,25.280786,22.732188,...,25.547651,25.354401,24.979038,25.520064,24.903071,24.253872,24.909618,24.885696,24.821770,25.250475
4,ENSG00000004975.12,ENSP00000005340.4,S170,RPRRRDSSEHGAGGH,1,23.247081,23.798530,25.622301,23.717474,22.842398,...,22.763420,22.372792,23.243067,21.957433,21.794215,22.454570,21.922218,23.618524,23.423444,22.315461
